In [1]:
import sys
import glob
import os, socket
from os.path import join
sys.path.append('../classifier')
import pandas as pd
import numpy as np
from torchvision import transforms
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from isic_artifact_classifier import ArtifactClassifier
from tqdm import tqdm

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_artifact = torch.load('../saved_models/isic-efficientnet-artifact/artifact_classifier.pth')
artifact_classifier = ArtifactClassifier()
artifact_classifier.load_state_dict(checkpoint_artifact['model_state_dict'])
artifact_classifier.eval()
artifact_classifier.to(DEVICE)
print('Loaded artifact classifier')



/home/amarkr/venvs/dent/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/amarkr/venvs/dent/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loaded artifact classifier


In [3]:
class CustomDataset(Dataset):
    def __init__(self, img_paths, transform=None):
        self.img_paths = img_paths 
        self.transform = transform
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, img_path

    def __len__(self):
        return len(self.img_paths)

transform = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])



## csv file for synthetic dataset

In [5]:
patients_info = {}
for disease in ['melanoma', 'nevus']:
    for STYLE in ['hairs', 'gel_bubbles', 'ruler', 'ink']:
        print(f'Processing {disease} with style: {STYLE} ...')
        IMAGE_FOLDER = f'/home/jupyter/dent_outputs/isic2019_v2/{disease}/{STYLE}'
        seeds = os.listdir(IMAGE_FOLDER)

        lambda_t_stars = [f'lambda_t_star{i}' for i in range(5, 46, 5)]

        style_image_paths = [join(IMAGE_FOLDER, seed, lambda_t_star, 'style.png') for seed in seeds for lambda_t_star in lambda_t_stars]
        style_image_paths = [path for path in style_image_paths if os.path.exists(path)]

        # Load the dataset
        dataset = CustomDataset(style_image_paths, transform=transform)
        test_loader = DataLoader(dataset, batch_size=64, num_workers=8, 
                                pin_memory=True)

        if STYLE == 'hairs':
            style_idx= 0
            OTHER_STYLES = ['gel_bubbles', 'ruler', 'ink']
            other_styles_idx = [1, 2, 3]
        elif STYLE == 'gel_bubbles':
            style_idx = 1
            OTHER_STYLES = ['hairs', 'ruler', 'ink']
            other_styles_idx = [0, 2, 3]
        elif STYLE == 'ruler':
            style_idx = 2
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ink']
            other_styles_idx = [0, 1, 3]
        elif STYLE == 'ink':
            style_idx = 3
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ruler']
            other_styles_idx = [0, 1, 2]

            
        for i, (img, img_path) in tqdm(enumerate(test_loader), total=len(test_loader)):
            img = img.to(DEVICE)
            with torch.no_grad():
                output = artifact_classifier(img) #output order is hairs, gel_bubble, ruler, ink
            output = torch.sigmoid(output)
            output = output.cpu().numpy()
            #loop over every image in the batch
            for j, path in enumerate(img_path):
                seed = path.split('/')[-3]
                lambda_t_star = path.split('/')[-2]
                
                if seed not in patients_info:
                    patients_info[seed] = {'seed': seed[4:], 'path': path}
                    
                    neutral_path = join(IMAGE_FOLDER, seed, 'lambda_t_star5', 'neutral.png')
                    img = Image.open(neutral_path).convert('RGB')
                    img = transform(img).unsqueeze(0).to(DEVICE)
                    with torch.no_grad():
                        neutral_output = artifact_classifier(img)
                    neutral_output = torch.sigmoid(neutral_output)
                    neutral_output = neutral_output.cpu().numpy()
                    patients_info[seed][f'neutral_{STYLE}'] = neutral_output[0][style_idx]
                    patients_info[seed][f'neutral_{OTHER_STYLES[0]}'] = neutral_output[0][other_styles_idx[0]]
                    patients_info[seed][f'neutral_{OTHER_STYLES[1]}'] = neutral_output[0][other_styles_idx[1]]
                    patients_info[seed][f'neutral_{OTHER_STYLES[2]}'] = neutral_output[0][other_styles_idx[2]]
                    
                patients_info[seed][f'{lambda_t_star[13:]}_{STYLE}'] = output[j][style_idx]
                patients_info[seed][f'{lambda_t_star[13:]}_{OTHER_STYLES[0]}'] = output[j][other_styles_idx[0]]
                patients_info[seed][f'{lambda_t_star[13:]}_{OTHER_STYLES[1]}'] = output[j][other_styles_idx[1]]
                patients_info[seed][f'{lambda_t_star[13:]}_{OTHER_STYLES[2]}'] = output[j][other_styles_idx[2]]

        df = pd.DataFrame.from_records(list(patients_info.values()), index='seed')
        df.to_csv(f'../metrics/cfrt/{disease}_{STYLE}_predictions.csv', index=True)   

Processing melanoma with style: hairs ...


100%|██████████| 141/141 [01:51<00:00,  1.26it/s]

Processing melanoma with style: gel_bubbles ...



100%|██████████| 141/141 [00:27<00:00,  5.13it/s]


Processing melanoma with style: ruler ...


...

100%|██████████| 141/141 [01:30<00:00,  1.57it/s]


Processing nevus with style: ruler ...


100%|██████████| 141/141 [01:29<00:00,  1.58it/s]


Processing nevus with style: ink ...


100%|██████████| 141/141 [01:28<00:00,  1.60it/s]


## csv file for the interpolated dataset

In [ ]:
patients_info = {}
for disease in ['melanoma', 'nevus']:
    for STYLE in ['hairs', 'gel_bubbles', 'ruler', 'ink']:
        print(f'Processing {disease} with style: {STYLE} ...')
        IMAGE_FOLDER = f'/home/jupyter/dent_outputs/interpolations/{disease}/{STYLE}'
        
        seeds = os.listdir(IMAGE_FOLDER)
        seeds = [int(seed.split('seed')[1]) for seed in seeds]
        frames = range(0, 50)

        style_image_paths = [join(IMAGE_FOLDER, f"seed{seed}", f'bezier_{seed}_{frame}.png') for seed in seeds for frame in frames]
        style_image_paths = [path for path in style_image_paths if os.path.exists(path)]

        # Load the dataset
        dataset = CustomDataset(style_image_paths, transform=transform)
        test_loader = DataLoader(dataset, batch_size=256, num_workers=8, 
                                pin_memory=True)

        if STYLE == 'hairs':
            style_idx= 0
            OTHER_STYLES = ['gel_bubbles', 'ruler', 'ink']
            other_styles_idx = [1, 2, 3]
        elif STYLE == 'gel_bubbles':
            style_idx = 1
            OTHER_STYLES = ['hairs', 'ruler', 'ink']
            other_styles_idx = [0, 2, 3]
        elif STYLE == 'ruler':
            style_idx = 2
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ink']
            other_styles_idx = [0, 1, 3]
        elif STYLE == 'ink':
            style_idx = 3
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ruler']
            other_styles_idx = [0, 1, 2]

            
        for i, (img, img_path) in tqdm(enumerate(test_loader), total=len(test_loader)):
            img = img.to(DEVICE)
            with torch.no_grad():
                output = artifact_classifier(img) #output order is hairs, gel_bubble, ruler, ink
            output = torch.sigmoid(output)
            output = output.cpu().numpy()
            #loop over every image in the batch
            for j, path in enumerate(img_path):
                seed = path.split('/')[-1].split('_')[1]
                frame = path.split('/')[-1].split('_')[2].split('.')[0]
                
                if seed not in patients_info:
                    patients_info[seed] = {'seed': seed}
                    
                patients_info[seed][f'{frame}_{STYLE}'] = output[0][style_idx]
                patients_info[seed][f'{frame}_{OTHER_STYLES[0]}'] = output[0][other_styles_idx[0]]
                patients_info[seed][f'{frame}_{OTHER_STYLES[1]}'] = output[0][other_styles_idx[1]]
                patients_info[seed][f'{frame}_{OTHER_STYLES[2]}'] = output[0][other_styles_idx[2]]
        print(a)
        df = pd.DataFrame.from_records(list(patients_info.values()), index='seed')
        df.to_csv(f'../metrics/cfrt/interpolations_{disease}_{STYLE}_predictions.csv', index=True)   

Processing melanoma with style: hairs ...


 16%|█▋        | 32/196 [00:28<01:02,  2.61it/s]

In [6]:
patients_info = {}
for disease in ['melanoma', 'nevus']:
    for STYLE in ['hairs', 'gel_bubbles', 'ruler', 'ink']:
        print(f'Processing {disease} with style: {STYLE} ...')
        IMAGE_FOLDER = f'/home/jupyter/dent_outputs/interpolations/{disease}/{STYLE}'
        
        # Check if directory exists
        if not os.path.exists(IMAGE_FOLDER):
            print(f"Directory {IMAGE_FOLDER} not found. Skipping...")
            continue
            
        seeds = os.listdir(IMAGE_FOLDER)
        seeds = [int(seed.split('seed')[1]) for seed in seeds]
        frames = range(0, 50)

        style_image_paths = [join(IMAGE_FOLDER, f"seed{seed}", f'bezier_{seed}_{frame}.png') 
                             for seed in seeds for frame in frames]
        style_image_paths = [path for path in style_image_paths if os.path.exists(path)]
        
        if not style_image_paths:
            print(f"No images found for {disease} with style {STYLE}. Skipping...")
            continue

        # Load the dataset
        dataset = CustomDataset(style_image_paths, transform=transform)
        test_loader = DataLoader(dataset, batch_size=256, num_workers=8, 
                                pin_memory=True)

        if STYLE == 'hairs':
            style_idx = 0
            OTHER_STYLES = ['gel_bubbles', 'ruler', 'ink']
            other_styles_idx = [1, 2, 3]
        elif STYLE == 'gel_bubbles':
            style_idx = 1
            OTHER_STYLES = ['hairs', 'ruler', 'ink']
            other_styles_idx = [0, 2, 3]
        elif STYLE == 'ruler':
            style_idx = 2
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ink']
            other_styles_idx = [0, 1, 3]
        elif STYLE == 'ink':
            style_idx = 3
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ruler']
            other_styles_idx = [0, 1, 2]
            
        for i, (img, img_path) in tqdm(enumerate(test_loader), total=len(test_loader)):
            img = img.to(DEVICE)
            with torch.no_grad():
                output = artifact_classifier(img)  # output order is hairs, gel_bubble, ruler, ink
            output = torch.sigmoid(output)
            output = output.cpu().numpy()
            
            # Loop over every image in the batch
            for j, path in enumerate(img_path):
                # Extract seed and frame from the path
                parts = path.split('/')
                filename = parts[-1]
                seed = filename.split('_')[1]
                frame = filename.split('_')[2].split('.')[0]
                
                if seed not in patients_info:
                    patients_info[seed] = {'seed': seed, 'disease': disease}
                    
                # Store predictions using the correct batch index j
                patients_info[seed][f'{frame}_{STYLE}'] = output[j][style_idx]
                patients_info[seed][f'{frame}_{OTHER_STYLES[0]}'] = output[j][other_styles_idx[0]]
                patients_info[seed][f'{frame}_{OTHER_STYLES[1]}'] = output[j][other_styles_idx[1]]
                patients_info[seed][f'{frame}_{OTHER_STYLES[2]}'] = output[j][other_styles_idx[2]]
        
        # Save intermediate results for each disease-style combination
        # This ensures we don't lose all data if the script crashes
        df = pd.DataFrame.from_records(list(patients_info.values()), index='seed')
        output_path = f'../metrics/cfrt/interpolations_{disease}_{STYLE}_predictions.csv'
        os.makedirs(os.path.dirname(output_path), exist_ok=True)  # Make sure directory exists
        df.to_csv(output_path, index=True)
        print(f"Saved predictions to {output_path}")

Processing melanoma with style: hairs ...


100%|██████████| 196/196 [02:30<00:00,  1.30it/s]


Saved predictions to ../metrics/cfrt/interpolations_melanoma_hairs_predictions.csv
Processing melanoma with style: gel_bubbles ...


100%|██████████| 196/196 [02:30<00:00,  1.30it/s]


Saved predictions to ../metrics/cfrt/interpolations_melanoma_gel_bubbles_predictions.csv
Processing melanoma with style: ruler ...


...

Saved predictions to ../metrics/cfrt/interpolations_nevus_gel_bubbles_predictions.csv
Processing nevus with style: ruler ...


100%|██████████| 196/196 [02:23<00:00,  1.37it/s]


Saved predictions to ../metrics/cfrt/interpolations_nevus_ruler_predictions.csv
Processing nevus with style: ink ...


100%|██████████| 196/196 [02:24<00:00,  1.36it/s]


Saved predictions to ../metrics/cfrt/interpolations_nevus_ink_predictions.csv


# Analysis 

In [7]:
for disease in ['melanoma', 'nevus']:
    for STYLE in ['hairs', 'gel_bubbles', 'ruler', 'ink']:
        print(f'Processing {disease} with style: {STYLE} ...')
        df = pd.read_csv(f'../metrics/cfrt/{disease}_{STYLE}_predictions.csv', index_col='seed')

        style_traj = ((df[f'5_{STYLE}'] - df[f'neutral_{STYLE}']) +
                    (df[f'10_{STYLE}'] - df[f'neutral_{STYLE}']) +
                    (df[f'15_{STYLE}'] - df[f'neutral_{STYLE}']) ) / 3
        
               
        if STYLE == 'hairs':
            style_idx= 0
            OTHER_STYLES = ['gel_bubbles', 'ruler', 'ink']
            other_styles_idx = [1, 2, 3]
        elif STYLE == 'gel_bubbles':
            style_idx = 1
            OTHER_STYLES = ['hairs', 'ruler', 'ink']
            other_styles_idx = [0, 2, 3]
        elif STYLE == 'ruler':
            style_idx = 2
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ink']
            other_styles_idx = [0, 1, 3]
        elif STYLE == 'ink':
            style_idx = 3
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ruler']
            other_styles_idx = [0, 1, 2]
        print('\tCurrent style:', STYLE)
        print('\tOther styles:', OTHER_STYLES)
        
        other_style_traj1 = ((df[f'5_{OTHER_STYLES[0]}'] - df[f'neutral_{OTHER_STYLES[0]}']) +
                    (df[f'10_{OTHER_STYLES[0]}'] - df[f'neutral_{OTHER_STYLES[0]}']) +
                    (df[f'15_{OTHER_STYLES[0]}'] - df[f'neutral_{OTHER_STYLES[0]}']) ) / 3
        
        other_style_traj2 = ((df[f'5_{OTHER_STYLES[1]}'] - df[f'neutral_{OTHER_STYLES[1]}']) +
                    (df[f'10_{OTHER_STYLES[1]}'] - df[f'neutral_{OTHER_STYLES[1]}']) +
                    (df[f'15_{OTHER_STYLES[1]}'] - df[f'neutral_{OTHER_STYLES[1]}']) ) / 3
        
        other_style_traj3 = ((df[f'5_{OTHER_STYLES[2]}'] - df[f'neutral_{OTHER_STYLES[2]}']) +
                    (df[f'10_{OTHER_STYLES[2]}'] - df[f'neutral_{OTHER_STYLES[2]}']) +
                    (df[f'15_{OTHER_STYLES[2]}'] - df[f'neutral_{OTHER_STYLES[2]}']) ) / 3
        
        condition1 = style_traj > other_style_traj1
        condition2 = style_traj > other_style_traj2
        condition3 = style_traj > other_style_traj3
        #Assign 1 if condition 1, 2, 3 are true
        cfrt_traj = condition1 & condition2 & condition3
        print('\t CFRT:', cfrt_traj.sum() / df.shape[0])
        #(sd_traj - pe_traj > 0).sum() / df.shape[0]



Processing melanoma with style: hairs ...
	Current style: hairs
	Other styles: ['gel_bubbles', 'ruler', 'ink']
	 CFRT: 0.907
Processing melanoma with style: gel_bubbles ...
	Current style: gel_bubbles
	Other styles: ['hairs', 'ruler', 'ink']
	 CFRT: 0.985
Processing melanoma with style: ruler ...
	Current style: ruler
	Other styles: ['hairs', 'gel_bubbles', 'ink']
	 CFRT: 0.721
Processing melanoma with style: ink ...
	Current style: ink
	Other styles: ['hairs', 'gel_bubbles', 'ruler']
	 CFRT: 0.412
Processing nevus with style: hairs ...
	Current style: hairs
	Other styles: ['gel_bubbles', 'ruler', 'ink']
	 CFRT: 0.855
Processing nevus with style: gel_bubbles ...
	Current style: gel_bubbles
	Other styles: ['hairs', 'ruler', 'ink']
	 CFRT: 0.973
Processing nevus with style: ruler ...
	Current style: ruler
	Other styles: ['hairs', 'gel_bubbles', 'ink']
	 CFRT: 0.956
Processing nevus with style: ink ...
	Current style: ink
	Other styles: ['hairs', 'gel_bubbles', 'ruler']
	 CFRT: 0.511


## analyze interpolations

In [9]:
for disease in ['melanoma', 'nevus']:
    for STYLE in ['hairs', 'gel_bubbles', 'ruler', 'ink']:
        print(f'Processing {disease} with style: {STYLE} ...')
        df = pd.read_csv(f'../metrics/cfrt/interpolations_{disease}_{STYLE}_predictions.csv', index_col='seed')

        style_cols = [f'{i}_{STYLE}' for i in range(30, 50)]
        style_traj = df[style_cols].sub(df[f'0_{STYLE}'], axis=0).mean(axis=1)
                       
        if STYLE == 'hairs':
            style_idx= 0
            OTHER_STYLES = ['gel_bubbles', 'ruler', 'ink']
            other_styles_idx = [1, 2, 3]
        elif STYLE == 'gel_bubbles':
            style_idx = 1
            OTHER_STYLES = ['hairs', 'ruler', 'ink']
            other_styles_idx = [0, 2, 3]
        elif STYLE == 'ruler':
            style_idx = 2
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ink']
            other_styles_idx = [0, 1, 3]
        elif STYLE == 'ink':
            style_idx = 3
            OTHER_STYLES = ['hairs', 'gel_bubbles', 'ruler']
            other_styles_idx = [0, 1, 2]
        print('\tCurrent style:', STYLE)
        print('\tOther styles:', OTHER_STYLES)
        
        other_style_cols1 = [f'{i}_{OTHER_STYLES[0]}' for i in range(30, 50)]
        other_style_traj1 = df[other_style_cols1].sub(df[f'0_{OTHER_STYLES[0]}'], axis=0).mean(axis=1)
        
        other_style_cols2 = [f'{i}_{OTHER_STYLES[1]}' for i in range(30, 50)]
        other_style_traj2 = df[other_style_cols2].sub(df[f'0_{OTHER_STYLES[1]}'], axis=0).mean(axis=1)
        
        other_style_cols3 = [f'{i}_{OTHER_STYLES[2]}' for i in range(30, 50)]
        other_style_traj3 = df[other_style_cols3].sub(df[f'0_{OTHER_STYLES[2]}'], axis=0).mean(axis=1)
        
        
        condition1 = style_traj > other_style_traj1
        condition2 = style_traj > other_style_traj2
        condition3 = style_traj > other_style_traj3
        #print('Style traj:', style_traj.mean())
        #print('Other style traj 1:', other_style_traj1.mean())
        #print('Other style traj 2:', other_style_traj2.mean())
        #print('Other style traj 3:', other_style_traj3.mean())
        #Assign 1 if condition 1, 2, 3 are true
        cfrt_traj = condition1 & condition2 & condition3
        print('\t CFRT:', cfrt_traj.sum() / df.shape[0])
        #(sd_traj - pe_traj > 0).sum() / df.shape[0]

Processing melanoma with style: hairs ...
	Current style: hairs
	Other styles: ['gel_bubbles', 'ruler', 'ink']
	 CFRT: 0.876
Processing melanoma with style: gel_bubbles ...
	Current style: gel_bubbles
	Other styles: ['hairs', 'ruler', 'ink']
	 CFRT: 0.992
Processing melanoma with style: ruler ...
	Current style: ruler
	Other styles: ['hairs', 'gel_bubbles', 'ink']
	 CFRT: 0.791
Processing melanoma with style: ink ...
	Current style: ink
	Other styles: ['hairs', 'gel_bubbles', 'ruler']
	 CFRT: 0.62
Processing nevus with style: hairs ...
	Current style: hairs
	Other styles: ['gel_bubbles', 'ruler', 'ink']
	 CFRT: 0.93
Processing nevus with style: gel_bubbles ...
	Current style: gel_bubbles
	Other styles: ['hairs', 'ruler', 'ink']
	 CFRT: 0.993
Processing nevus with style: ruler ...
	Current style: ruler
	Other styles: ['hairs', 'gel_bubbles', 'ink']
	 CFRT: 0.975
Processing nevus with style: ink ...
	Current style: ink
	Other styles: ['hairs', 'gel_bubbles', 'ruler']
	 CFRT: 0.682
